# Wordle Solver

## Introduction

I started working on this project when I was first learning python and wanted to get better at list manipulation. Like everyone else I know, I play Wordle and figured that might a good space to practice, so I decided to build a function to play Wordle. And, once I had the general framework built, I could play around with it to make further improvements. 


## Table of Contents

* [Wordle Solver v0](#first-bullet)
* [Wordle Solver v1](#second-bullet)
* [Wordle Solver v2](#second-bullet)
* [Wordle Solver v3](#second-bullet)

In [1]:
import pandas as pd
import numpy  as np
import string
import random
import math


## list of words

wordle_answers = pd.read_csv("wordle_answers.txt", header = None).squeeze().tolist()
wordle_guesses = pd.read_csv("wordle_guesses.txt", header = None).squeeze().tolist()


## Wordle Solver v0 <a class="anchor" id="first-bullet"></a>

Below is the code for our initial Wordle solver. The function takes two arguments: the solution and a list of potential words. And the function returns the solution, an indicator for whether or not the puzzle was solved, the number of attempts the function made, the word guessed for each attempt, and the number of words in the word pool after each attempt. We can use these outputs to assess and improve the function down the road.

The function itself just contains the logic for the basic Wordle gameplay. The function makes up to six attempts to guess the solution in an iterative process. After each attempt, the feedback rules are applied to the guessed word and the remaining word pool is winnowed to remove any words that contradict the feedback. The feedback rules are applied in the strictest sense, such that the function won't consider words that don't contain a "green" letter in the correct place, don't contain a "yellow" letter in a place other than what attempted, or contain any "grey" letters. In effect, the function is playing Wordle on "hard mode." However, beyond that, the function doesn't contain any additional logic; it just picks a word from the word pool at random. 

*NOTE: Wordle's list of potential answers contains roughly 2,300 words, but there are nearly 15,000 words that it will accept as guess. We will only be drawing solutions from the list of 2,300 words, but our guesses will come from the full list of 15,000 words.

In [ ]:
def wordle_solver_0(solution, word_list):
    
    solution = solution.upper()

    ## initializing some lists to hold our guesses
    solved  = "N"
    n_tries = 6 
    guesses = []
    remaining_words = []

    ## making sure the correct answer is in our lexicon
    if solution not in word_list:

        print("Not a valid word")
    
    ## if it is a valid word, we attempt to 
    else:

        ## Looping through our guesses
        for i in range(1, 7):

            ## guessing a word at random
            temp_guess = random.choice(word_list)

            if temp_guess == solution:

                ## if we guess correctly, break the loop
                guesses.append(temp_guess)
                solved  = "Y"
                n_tries = i
                break

            else:
                ## determining the "color" of each letter in our guess
                # green letters have to match the location in the solution
                temp_green  = [a if a == b else '*' for a, b in zip(temp_guess, solution)]
                # yellow letters have to be the solution, but in the wrong place, and not count a double letter if the solution doesn't contain a double letter
                temp_yellow = [a if a in solution else '*' for a in temp_guess]
                temp_yellow = [a if a != b and a in solution else '*' for a, b in zip(temp_guess, solution)]
                    # note: this one was all chatGPT. Trying to figure this one out broke my brain
                temp_yellow = [a if sum(1 for i, l in enumerate(temp_yellow[:j]) if l == a) + temp_green.count(a) < solution.count(a) else '*' for j, a in enumerate(temp_yellow)]                
                # grey letters aren't in the solution and don't match the yellow letter
                temp_grey = [a if a != b else '*' for a, b in zip(temp_guess, solution)]
                temp_grey = [a if a != b else '*' for a, b in zip(temp_grey, temp_yellow)]

                ## updating our potential word bank
                # eliminating any word that does not match temp_green
                word_list = [word for word in word_list if all(a == b or b == '*' for a, b in zip(list(word), temp_green))]
                # eliminating any word that matches temp_yellow, but contains letters that appear the correct number of times (temp_green+temp_yellow)
                word_list = [word for word in word_list if all(a != b or b == "*" for a, b in zip(word, temp_yellow))]
                word_list = [word for word in word_list if all((temp_green + temp_yellow).count(l) <= word.count(l) for l in [a for a in set(temp_green + temp_yellow) if a != '*'])]
                # eliminating any word that matches temp_grey
                word_list = [word for word in word_list if all(a != b or b == "*"  for a, b in zip(word, temp_grey))]
                word_list = [word for word in word_list if not any(l in word for l in [a for a in temp_grey if a not in (temp_green + temp_yellow)])]

                ## appending our guess
                guesses.append(temp_guess)
                remaining_words.append(len(word_list))

        return [solution, solved, n_tries, guesses, remaining_words]


Below we loop over the potential Wordle solutions to assess our function. The function is able to solve the puzzle 87% of the time and, when it solves the puzzles, it takes 4.65 guesses on average to do so. Not great, but this is a naive model - meaning it's just guessing at random - and meant to serve as a baseline. 

In [ ]:
random.seed(5)
results_0  = []

for i in wordle_answers:
    
    temp_result = wordle_solver_0(solution = i, word_list = wordle_guesses)

    results_0.append(temp_result)


## converted results into a data frame

df_results_0 = pd.DataFrame({'solution'        : [a[0] for a in results_0],
                             'solved'          : [a[1] for a in results_0],
                             'n_tries'         : [a[2] for a in results_0],
                             'guesses'         : [a[3] for a in results_0],
                             'remaining_words' : [a[4] for a in results_0]})


## calculating basic stats

print(round(np.mean(df_results_0['solved'] == 'Y'), 3))
print(round(np.mean(df_results_0.loc[df_results_0['solved'] == 'Y', ['n_tries']]), 2))



0.872
4.65


## Wordle Solver v1 <a class="anchor" id="second-bullet"></a>

A quick inspection of the guesses made during the failed attempts of our initial function highlight a glaring issue: poor first guess strategy. Words like YLEMS, MOROR, KYACK probably are great openers. They either contain uncommon letters or duplicate letters - neither of which are likely to provide much information. Improving the first guess could be an easy way to improve our function. 

In [4]:
## guesses made during failed attempts

df_results_0.loc[df_results_0['solved'] == 'N']['guesses'].head(20)

21     [YLEMS, OTHER, GROPE, KIORE, ABORE, AFORE]
28     [EYRIR, UDDER, LONER, WATER, ARTER, ATTER]
34     [MATHS, AYGRE, APING, AWING, AXING, AKING]
46     [VANGA, AREDE, ASCUS, ALLOT, ALKYL, ALMAH]
63     [MIGMA, BEPAT, CATER, ATTER, ASTER, ARTER]
88     [GROVY, FORDS, HORAH, NORIA, TORTA, PORTA]
95     [LEESE, COADY, AMIGO, AFOOT, ABHOR, ARROZ]
141    [BOOTH, BURAN, BABKA, BAILS, BAAED, BAMMY]
145    [SEPTS, MURID, LAGGY, HALON, CANAL, KANAL]
175    [HEVEL, LEEZE, PERLE, MELOE, TELAE, WELKE]
179    [PILOT, EANED, RUNCE, KENCH, HENCH, WENCH]
181    [GORED, PURRE, SERRY, MERRY, VERRY, FERRY]
184    [KYACK, OUPED, FILER, GLEES, NEWEL, HEVEL]
186    [CURNY, SMITH, APEAK, WODGE, JEBEL, BEVEL]
197    [AMIDS, CONIC, VIPER, RIFFY, LIBRI, BIRZZ]
202    [YOWIE, CASTE, ARERE, NUGAE, PEAKE, EVADE]
206    [YAJNA, AKSED, HUIAS, SCART, LOAST, PLAST]
216    [MOROR, CUFFS, NINNY, AKING, KWINK, DEINK]
233    [REJON, CLOPS, FOIST, MOUST, HOAST, TOAST]
238    [ORLOP, HOOKA, DOOFS, BOOMY, BOOZY, BOOGY]


Below we calculate the percent of words that contain each letter in the alphabet and display the top and bottom ten most frequently used letters. Not surprisingly, Q, X, J, Z, and V are among the least likely to appear in a word. 

In [5]:
## determining which letters appears in each word

df_letter_freq = pd.DataFrame({'word': wordle_guesses})

for i in string.ascii_uppercase:

    df_letter_freq[i] = df_letter_freq['word'].str.contains(i)

## reshaping data

df_letter_freq = df_letter_freq.melt(id_vars    = ['word'],
                                     value_vars = list(string.ascii_uppercase),
                                     var_name   = 'letter',
                                     value_name = 'contains')


## calculated the frequency of each letter

df_letter_freq = (df_letter_freq
                  .groupby('letter')
                  .agg(freq  = ('contains', lambda x: round((x == True).mean(), 2)))
                  .sort_values('freq', ascending = False)
                  )

print(df_letter_freq.head(10))
print(df_letter_freq.tail(10))



        freq
letter      
S       0.44
E       0.43
A       0.42
O       0.31
R       0.30
I       0.28
L       0.23
T       0.23
N       0.22
U       0.19
        freq
letter      
G       0.12
B       0.12
K       0.11
W       0.08
F       0.07
V       0.05
Z       0.03
J       0.02
X       0.02
Q       0.01


I was also curious if attempts where the first guess contained a duplicate letter (e.g. BOOTH, PROMO) were less likely to be solved. It's not a huge different, but we were about 1.5% less likely to solve the puzzle when the initial guess contained a duplicate letter. 

In [11]:
## seeing if the first guess has a repeated letter
df_results_0['first_guess'] = [a[3][0] for a in results_0]
df_results_0['dup_letter']  = df_results_0['first_guess'].apply(lambda x: (len(set(x)) == 5) == False)

(df_results_0
.groupby('dup_letter')
.agg(count  = ('dup_letter', 'size'),
     solved = ('solved', lambda x: round((x == 'Y').mean(), 3)))
)


,count,solved
dup_letter,,
False,1446,0.878
True,865,0.862


Below we made two small changes to the guessing logic of our function. First, the initial guess must be comprised of the ten most common letters without repeats - winnowing the initial guess pool from 15,000 to 540 words. Second, during subsequent attempts, the guess must contain the maximum number of unique letters possible - preventing duplicate letters from being guessed *unless* we know the solution contains a duplicate letter.

In [ ]:
def wordle_solver_1(solution, word_list):
    
    solution = solution.upper()

    ## initializing some lists to hold our guesses
    solved  = "N"
    n_tries = 6 
    guesses = []
    remaining_words = []

    ## making sure the correct answer is in our lexicon
    if solution not in word_list:

        print("Not a valid word")
    
    ## if it is a valid word, we attempt to 
    else:

        ## Looping through our guesses
        for i in range(1, 7):

            ## first guess
            if i == 1:
                
                temp_guess = word_list.copy()
                temp_guess = [x for x in temp_guess if sum(l in 'SEAORILTNU' for l in set(x)) == 5]
                temp_guess = random.choice(temp_guess)

            ## subsequent guesses
            else:

                temp_guess = word_list.copy()

                ## selecting the words that contain the most unique letters
                
                temp_max_letters = max([len(set(y)) for y in temp_guess])
                temp_guess       = [x for x in temp_guess if len(set(x)) == temp_max_letters]   

                ## guessing a word at random
                temp_guess = random.choice(temp_guess)

            ## checking to see if we guessed correctly
            if temp_guess == solution:

                ## if we guess correctly, break the loop
                guesses.append(temp_guess)
                solved  = "Y"
                n_tries = i
                break

            else:
                ## determining the "color" of each letter in our guess
                # green letters have to match the location in the solution
                temp_green  = [a if a == b else '*' for a, b in zip(temp_guess, solution)]
                # yellow letters have to be the solution, but in the wrong place, and not count a double letter if the solution doesn't contain a double letter
                temp_yellow = [a if a in solution else '*' for a in temp_guess]
                temp_yellow = [a if a != b and a in solution else '*' for a, b in zip(temp_guess, solution)]
                    # note: this one was all chatGPT. Trying to figure this one out broke my brain
                temp_yellow = [a if sum(1 for i, l in enumerate(temp_yellow[:j]) if l == a) + temp_green.count(a) < solution.count(a) else '*' for j, a in enumerate(temp_yellow)]                
                # grey letters aren't in the solution and don't match the yellow letter
                temp_grey = [a if a != b else '*' for a, b in zip(temp_guess, solution)]
                temp_grey = [a if a != b else '*' for a, b in zip(temp_grey, temp_yellow)]

                ## updating our potential word bank
                # eliminating any word that does not match temp_green
                word_list = [word for word in word_list if all(a == b or b == '*' for a, b in zip(list(word), temp_green))]
                # eliminating any word that matches temp_yellow, but contains letters that appear the correct number of times (temp_green+temp_yellow)
                word_list = [word for word in word_list if all(a != b or b == "*" for a, b in zip(word, temp_yellow))]
                word_list = [word for word in word_list if all((temp_green + temp_yellow).count(l) <= word.count(l) for l in [a for a in set(temp_green + temp_yellow) if a != '*'])]
                # eliminating any word that matches temp_grey
                word_list = [word for word in word_list if all(a != b or b == "*"  for a, b in zip(word, temp_grey))]
                word_list = [word for word in word_list if not any(l in word for l in [a for a in temp_grey if a not in (temp_green + temp_yellow)])]

                ## appending our guess
                guesses.append(temp_guess)
                remaining_words.append(len(word_list))

        return [solution, solved, n_tries, guesses, remaining_words]


Again, we loop over the list of Wordle solutions to test our new function. The result is that our solve rate has increased to 90.5% - a three percent increase. It's not a huge lift - but it's a start. 

In [ ]:
random.seed(5)
results_1  = []

for i in wordle_answers:
    
    temp_result = wordle_solver_1(solution = i, word_list = wordle_guesses)

    results_1.append(temp_result)


## converted results into a data frame

df_results_1 = pd.DataFrame({'solution'        : [a[0] for a in results_1],
                             'solved'          : [a[1] for a in results_1],
                             'n_tries'         : [a[2] for a in results_1],
                             'guesses'         : [a[3] for a in results_1],
                             'remaining_words' : [a[4] for a in results_1]})


## calculating basic stats

print(round(np.mean(df_results_1['solved'] == 'Y'), 3))
print(round(np.mean(df_results_1.loc[df_results_1['solved'] == 'Y', ['n_tries']]), 2))


0.905
4.44


## Wordle Solver v2 <a class="anchor" id="third-bullet"></a>

Building on the logic in v1, we can change the guessing logic for subsequent attempts to ensure that we are choosing a word that contains the letters most likely to be in the solution. Do to that, we will need to create a function to calculate the frequency of letters within our remaining word pool. 

In [16]:
## defining a feature to return the most frequent letters our word list

def letter_freq(all_words):

    letter_freq = pd.DataFrame({'word': all_words})

    for i in string.ascii_uppercase:

            letter_freq[i] = letter_freq['word'].str.contains(i)

    letter_freq = (
                    letter_freq
                    .melt(value_vars = list(string.ascii_uppercase),
                        var_name   = 'letter',
                        value_name = 'contains')
                    .groupby('letter')
                    .agg(freq_pct = ('contains', lambda x: round((x == True).mean(), 3)))
                    .sort_values('freq_pct', ascending = False)
                    .reset_index()
                    )

    return letter_freq


We can then insert the letter frequency function into the Wordle solver. Now, during subsequent guesses, we will first filter out words that do not contain the most frequently used letter within our word pool. Then, we recalculate the letter frequency for the new word pool and filter out any words that contain the least frequently used letter. In both cases, the frequency has to be between 1 and 0. That is, we aren't going to rule out words containing a letter with a frequency of 0 - because those words have already been ruled out.

In [ ]:
def wordle_solver_2(solution, word_list):
    
    solution  = solution.upper()
    word_list = word_list.copy()

    ## initializing some lists to hold our guesses
    solved  = "N"
    n_tries = 6 
    guesses = []
    remaining_words = []

    ## making sure the correct answer is in our lexicon
    if solution not in word_list:

        print("Not a valid word")
    
    ## if it is a valid word, we attempt to 
    else:

        ## Looping through our guesses
        for i in range(1, 7):

            ## first guess
            if i == 1:
                
                temp_guess = word_list.copy()
                temp_guess = [x for x in temp_guess if sum(l in 'SEAORILTNU' for l in set(x)) == 5]
                temp_guess = random.choice(temp_guess)

            ## subsequent guesses
            else:
                
                temp_guess = word_list.copy()

                ## eliminating words that don't contain the most likely letter
                if len(temp_guess) > 1:

                    temp_letter_freq = letter_freq(all_words = temp_guess)
                    temp_letter_high = temp_letter_freq[temp_letter_freq['freq_pct'] < 1]
                    temp_letter_high = temp_letter_high[temp_letter_high['freq_pct'] > 0]
                
                    if len(temp_letter_high) > 0:

                        temp_letter_high = temp_letter_high.loc[lambda df: df['freq_pct'].idxmax()]['letter']
                        temp_guess       = [x for x in temp_guess if any(l in x for l in temp_letter_high) == True]

                ## eliminating words that contain the least likely letter
                if len(temp_guess) > 1:

                    temp_letter_freq = letter_freq(all_words = temp_guess)
                    temp_letter_low  = temp_letter_freq[temp_letter_freq['freq_pct'] < 1]
                    temp_letter_low  = temp_letter_low[temp_letter_low['freq_pct'] > 0]
                    
                    if len(temp_letter_low) > 0:

                        temp_letter_low = temp_letter_low.loc[lambda df: df['freq_pct'].idxmin()]['letter']
                        temp_guess      = [x for x in temp_guess if any(l in x for l in temp_letter_low) == False]
                
                ## selecting the words that contain the most unique letters
                
                temp_max_letters = max([len(set(y)) for y in temp_guess])
                temp_guess       = [x for x in temp_guess if len(set(x)) == temp_max_letters]       

                ## picking a word at random from the remaining pool
                temp_guess = random.choice(temp_guess)

            ## checking to see if we guessed correctly
            if temp_guess == solution:

                ## if we guess correctly, break the loop
                guesses.append(temp_guess)
                solved  = "Y"
                n_tries = i
                break

            else:
                ## determining the "color" of each letter in our guess
                # green letters have to match the location in the solution
                temp_green  = [a if a == b else '*' for a, b in zip(temp_guess, solution)]
                # yellow letters have to be the solution, but in the wrong place, and not count a double letter if the solution doesn't contain a double letter
                temp_yellow = [a if a in solution else '*' for a in temp_guess]
                temp_yellow = [a if a != b and a in solution else '*' for a, b in zip(temp_guess, solution)]
                    # note: this one was all chatGPT. Trying to figure this one out broke my brain
                temp_yellow = [a if sum(1 for i, l in enumerate(temp_yellow[:j]) if l == a) + temp_green.count(a) < solution.count(a) else '*' for j, a in enumerate(temp_yellow)]                
                # grey letters aren't in the solution and don't match the yellow letter
                temp_grey = [a if a != b else '*' for a, b in zip(temp_guess, solution)]
                temp_grey = [a if a != b else '*' for a, b in zip(temp_grey, temp_yellow)]

                ## updating our potential word bank
                # eliminating any word that does not match temp_green
                word_list = [word for word in word_list if all(a == b or b == '*' for a, b in zip(list(word), temp_green))]
                # eliminating any word that matches temp_yellow, but contains letters that appear the correct number of times (temp_green+temp_yellow)
                word_list = [word for word in word_list if all(a != b or b == "*" for a, b in zip(word, temp_yellow))]
                word_list = [word for word in word_list if all((temp_green + temp_yellow).count(l) <= word.count(l) for l in [a for a in set(temp_green + temp_yellow) if a != '*'])]
                # eliminating any word that matches temp_grey
                word_list = [word for word in word_list if all(a != b or b == "*"  for a, b in zip(word, temp_grey))]
                word_list = [word for word in word_list if not any(l in word for l in [a for a in temp_grey if a not in (temp_green + temp_yellow)])]

                ## appending our guess
                guesses.append(temp_guess)
                remaining_words.append(len(word_list))

        return [solution, solved, n_tries, guesses, remaining_words]


After looping over the possible Wordle solutions, we can see that our solve rate has increased - but only by 2% to 92.5%. That's not exactly the gain I had hoped for. Granted, I could iterate this process to continue to prune the word pool, but I'm not sure if that is the best and/or most efficient approach.

In [ ]:
random.seed(5)
results_2  = []

for i in wordle_answers:
    
    temp_result = wordle_solver_2(solution = i, word_list = wordle_guesses)

    results_2.append(temp_result)


## converted results into a data frame

df_results_2 = pd.DataFrame({'solution'        : [a[0] for a in results_2],
                             'solved'          : [a[1] for a in results_2],
                             'n_tries'         : [a[2] for a in results_2],
                             'guesses'         : [a[3] for a in results_2],
                             'remaining_words' : [a[4] for a in results_2]})


## calculating basic stats

print(round(np.mean(df_results_2['solved'] == 'Y'), 3))
print(round(np.mean(df_results_2.loc[df_results_2['solved'] == 'Y', ['n_tries']]), 2))

0.925
4.4


Another visual inspection of the failed attempts provides some hints. The function actually does a pretty good job of winnowing the word pool in the first couple of rounds, but then it stalls. In many cases, after winnowing the word pool to less than ten words - the remaining attempts only remove one word at a time. The function falls into a trap where all but one letter is solved, but there are several letters that could fit (e.g. JUDGE, FUDGE, NUDGE for BUDGE). As a result, I don't think focusing on letter frequency is likely to help. Rather, we need a strategy that would avoid these situations entirely. 

In [21]:
df_results_2.loc[df_results_2['solved'] == 'N'][['solution', 'guesses', 'remaining_words']].head(25)

,solution,guesses,remaining_words
7,ABOVE,"[AISLE, ADOZE, ACONE, AFORE, AWOKE, AMOVE]","[57, 12, 8, 5, 3, 1]"
34,AGING,"[TOLAR, WAQFS, CHAVE, AUXIN, APING, AKING]","[1480, 274, 60, 6, 2, 1]"
98,ARDOR,"[LAIRS, CREAK, WRATH, GROMA, ARROZ, ARBOR]","[396, 68, 10, 3, 2, 1]"
136,AZURE,"[LARES, EXTRA, DEARN, AMORE, AFIRE, AYGRE]","[185, 36, 10, 4, 3, 2]"
146,BANJO,"[ROUST, OLIVA, AGONE, NACHO, MANZO, PANKO]","[554, 95, 18, 3, 2, 1]"
155,BATCH,"[STEAN, CLART, MATCH, GATCH, PATCH, WATCH]","[309, 18, 5, 4, 3, 2]"
177,BELLY,"[TRIOL, UHLAN, YELPS, CELLY, DELLY, FELLY]","[1001, 57, 8, 7, 6, 5]"
181,BERRY,"[UNITS, LEACH, PERVO, JERKY, MERRY, DERRY]","[2587, 151, 18, 6, 3, 2]"
186,BEZEL,"[ISNAE, REPLY, METOL, HEVEL, JEBEL, BEDEL]","[951, 43, 11, 6, 2, 1]"
193,BINGE,"[NEALS, CODEN, TINGE, HINGE, MINGE, PINGE]","[299, 66, 7, 6, 5, 4]"


## Wordle Solver v3 <a class="anchor" id="fourth-bullet"></a>

I did some research and decided to use entropy to guide the guessing logic, rather the focusing on letters most or least likely to appear. There are better explanations of entropy on the web, but basically this process calculates the information gained from each potential guess. In this case, it works by calculating the number and frequency of unique outcomes (i.e. Wordle feedback patterns) that you could get from guessing a particular word given the all of the potential solutions. By selecting the word with the greatest entropy, you can winnow the field much faster. Or put another way, you are trying to eliminate as many solutions as possible rather than attempting to guess the correct answer. 

Below is our entropy function, which takes two arguments: a word and a list of words. The function compares the word to each word in the word pool to determine the Wordle feedback pattern and stores the pattern in a list. Then we loop over the unique patterns and calculate its frequency (i.e. number of times that pattern appears/total number of patterns) and using a summation function, calculates the word's entropy.*

*It's really easy to understand how maximizing entropy improves performance when you break down this formula. Entropy is higher when (a) there are more unique feedback patterns and/or (b) the distribution of pattern frequencies is more uniform. Meaing the ideal word to guess is one that fractures the possible solutions into several, equally sized bins. This should help avoid guessing a word that could lead to a situation where all but one letter is solved, but the remaining letter could have several potential values (e.g. S_OOP - SCOOP, SWOOP, STOOP, etc.). 

In [35]:
def word_entropy(word, all_words):
    
    possible_patterns = []
    
    ## Feedback Loop
    for word_2 in all_words:
    
        ## Feedback
        temp_feedback = ['B']*5

        for j in range(5):
            if word[j] == word_2[j]:
                temp_feedback[j] = 'G'
            if word[j] in [y if x != y else '*' for x, y in zip(word, word_2)]:
                temp_feedback[j] = 'Y'
                
        temp_feedback = ''.join(temp_feedback)
        possible_patterns.append(temp_feedback)
    
    ## Calculating Entropy
    unique_patterns = list(set(possible_patterns))
    entropy         = 0.0
    
    for pattern in unique_patterns:
        
        temp_prob = sum([1 if x == pattern else 0 for x in possible_patterns])/len(possible_patterns)
        
        entropy -= temp_prob*math.log2(temp_prob)
        
    return entropy

With the entropy function defined, we can slide it into our Wordle solver function. At the start of each attempt, we will iterate over all of the words in the word pool and calculate their entropy. And, as we iterate over that list, we store the word with the greatest entropy as a variable. And, after the entropy loop is done, we can then use that variable as our guess. Additionally, I iterated the entropy function over the entire list of potential guesses and it identified 'TARES' as the best first guess - so I hard coded that into the function. 

In [ ]:

def wordle_solver_3(solution, word_list):
    
    solution  = solution.upper()
    word_list = word_list.copy()

    ## initializing some lists to hold our guesses
    solved  = "N"
    n_tries = 6 
    guesses = []
    remaining_words = []

    ## making sure the correct answer is in our lexicon
    if solution not in word_list:

        print("Not a valid word")
    
    ## if it is a valid word, we attempt to 
    else:

        ## Looping through our guesses
        for i in range(1, 7):

            ## first guess
            if i == 1:
                
                temp_guess = 'TARES'

            ## subsequent guesses
            elif len(word_list) > 1:
                
                best_entropy = 0.0                
                
                for j in word_list:
                    
                    temp_entropy = word_entropy(word = j, all_words = word_list)
                    
                    if temp_entropy > best_entropy:
                        
                        best_entropy = temp_entropy   
                        temp_guess   = j
                        
            else:
                
                temp_guess = word_list[0]

            ## checking to see if we guessed correctly
            if temp_guess == solution:

                ## if we guess correctly, break the loop
                guesses.append(temp_guess)
                solved  = "Y"
                n_tries = i
                break

            else:
                ## determining the "color" of each letter in our guess
                # green letters have to match the location in the solution
                temp_green  = [a if a == b else '*' for a, b in zip(temp_guess, solution)]
                # yellow letters have to be the solution, but in the wrong place, and not count a double letter if the solution doesn't contain a double letter
                temp_yellow = [a if a in solution else '*' for a in temp_guess]
                temp_yellow = [a if a != b and a in solution else '*' for a, b in zip(temp_guess, solution)]
                    # note: this one was all chatGPT. Trying to figure this one out broke my brain
                temp_yellow = [a if sum(1 for i, l in enumerate(temp_yellow[:j]) if l == a) + temp_green.count(a) < solution.count(a) else '*' for j, a in enumerate(temp_yellow)]                
                # grey letters aren't in the solution and don't match the yellow letter
                temp_grey = [a if a != b else '*' for a, b in zip(temp_guess, solution)]
                temp_grey = [a if a != b else '*' for a, b in zip(temp_grey, temp_yellow)]

                ## updating our potential word bank
                # eliminating any word that does not match temp_green
                word_list = [word for word in word_list if all(a == b or b == '*' for a, b in zip(list(word), temp_green))]
                # eliminating any word that matches temp_yellow, but contains letters that appear the correct number of times (temp_green+temp_yellow)
                word_list = [word for word in word_list if all(a != b or b == "*" for a, b in zip(word, temp_yellow))]
                word_list = [word for word in word_list if all((temp_green + temp_yellow).count(l) <= word.count(l) for l in [a for a in set(temp_green + temp_yellow) if a != '*'])]
                # eliminating any word that matches temp_grey
                word_list = [word for word in word_list if all(a != b or b == "*"  for a, b in zip(word, temp_grey))]
                word_list = [word for word in word_list if not any(l in word for l in [a for a in temp_grey if a not in (temp_green + temp_yellow)])]

                ## appending our guess
                guesses.append(temp_guess)
                remaining_words.append(len(word_list))

        return [solution, solved, n_tries, guesses, remaining_words]



Using the entropy strategy, our solve rate has increased to 95%. And when solved, the function only takes 4.2 attempts on average. Granted, having to calculate entropy for every word in the word pool for each attempt has drastically increased run time. 

In [ ]:
random.seed(5)
results_3  = []

for i in wordle_answers:
    
    temp_result = wordle_solver_3(solution = i, word_list = wordle_guesses)

    results_3.append(temp_result)
    
    print(i)


## converted results into a data frame

df_results_3 = pd.DataFrame({'solution'        : [a[0] for a in results_3],
                             'solved'          : [a[1] for a in results_3],
                             'n_tries'         : [a[2] for a in results_3],
                             'guesses'         : [a[3] for a in results_3],
                             'remaining_words' : [a[4] for a in results_3]})


## calculating basic stats

print(round(np.mean(df_results_3['solved'] == 'Y'), 3))
print(round(np.mean(df_results_3.loc[df_results_3['solved'] == 'Y', ['n_tries']]), 2))


0.952
4.2


And, below we can inspect the solutions that weren't solved by our function. It still gets stuck in those one-letter-multiple-solution-traps, but generally does better than the letter frequency method.

In [44]:
df_results_3.loc[df_results_3['solved'] == 'N'][['solution', 'guesses', 'remaining_words']].head(30)

,solution,guesses,remaining_words
34,AGING,"[TARES, ALOIN, ACING, AHING, AKING, APING]","[726, 13, 6, 5, 4, 3]"
141,BAGGY,"[TARES, MANLY, BAWDY, BACCY, BAFFY, BABBY]","[521, 48, 5, 3, 2, 1]"
238,BOOBY,"[TARES, NOILY, BODGY, BOOFY, BOOKY, BOOMY]","[1022, 75, 7, 4, 3, 2]"
243,BOOZY,"[TARES, NOILY, BODGY, BOOFY, BOOKY, BOOMY]","[1022, 75, 7, 4, 3, 2]"
253,BOXER,"[TARES, DOPER, MOWER, GONER, YOKER, COVER]","[307, 39, 22, 14, 8, 3]"
260,BRASS,"[TARES, BRAGS, BRADS, BRAHS, BRAKS, BRANS]","[111, 7, 6, 5, 4, 3]"
261,BRAVE,"[TARES, BEARD, BRACE, BRAKE, BRAME, BRANE]","[169, 6, 5, 4, 3, 2]"
336,CATCH,"[TARES, HAINT, BATCH, GATCH, LATCH, MATCH]","[131, 11, 6, 5, 4, 3]"
363,CHESS,"[TARES, DEILS, OPENS, CHEWS, CHEFS, CHEMS]","[316, 50, 16, 3, 2, 1]"
398,CLASS,"[TARES, MOALS, CLAPS, CLADS, CLAGS, CLANS]","[378, 39, 6, 5, 4, 3]"


## Final Remarks

I think I can still improve upon v3's performance. The "yellow" letter logic in the entropy function doesn't handle duplicate letters perfectly, but I wanted to test it before sinking a ton of time into it. I could also cut down the runtime significantly if only iterated over the first 37% of the randomized word pool, consistent with the odds algorithm used to solve the secretary problem. 